# InstructGPT RLHF Walkthrough

Goal: run the smallest honest tensor version of SFT, reward-model training, and PPO-style policy optimization.


## Paper-to-code map

Official artifact:

- `learning/rlhf-classic/official/repos/following-instructions-human-feedback/README.md`
- `learning/rlhf-classic/official/repos/following-instructions-human-feedback/model-card.md`
- `learning/rlhf-classic/official/repos/following-instructions-human-feedback/automatic-eval-samples/*.csv`

Runnable local source:

- `learning/rlhf-classic/src/sft_minimal.py`
- `learning/rlhf-classic/src/rm_minimal.py`
- `learning/rlhf-classic/src/ppo_llm_minimal.py`


In [ ]:
from __future__ import annotations
import importlib.util
import math
import platform
from pathlib import Path
import torch
import torch.nn.functional as F
REPO = Path.cwd()
if not (REPO / "learning").exists():
    REPO = Path.cwd().parent
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("repo", REPO)


## Step 1: SFT as masked next-token loss


In [ ]:
torch.manual_seed(0)
B, T, V = 2, 6, 11
input_ids = torch.randint(0, V, (B, T))
logits = torch.randn(B, T - 1, V)
targets = input_ids[:, 1:]
response_mask = torch.tensor([[0, 0, 1, 1, 1], [0, 1, 1, 1, 1]], dtype=torch.float32)
per_token_loss = F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1), reduction="none").view(B, T - 1)
sft_loss = (per_token_loss * response_mask).sum() / response_mask.sum()
print("sft_loss", round(float(sft_loss), 4))


## Step 2: reward model Bradley-Terry loss


In [ ]:
r_chosen = torch.tensor([1.2, -0.1, 0.7])
r_rejected = torch.tensor([0.4, 0.2, -0.3])
rm_loss = -F.logsigmoid(r_chosen - r_rejected).mean()
rm_acc = (r_chosen > r_rejected).float().mean()
print("rm_loss", round(float(rm_loss), 4), "rm_acc", round(float(rm_acc), 4))


## Step 3: PPO clipped objective and KL intuition


In [ ]:
logp_old = torch.tensor([-1.0, -0.8, -1.4, -0.9])
logp_new = torch.tensor([-0.9, -1.2, -1.0, -0.7])
advantages = torch.tensor([0.5, 0.7, -0.4, 1.2])
ratio = (logp_new - logp_old).exp()
ppo_loss = -torch.min(ratio * advantages, ratio.clamp(0.8, 1.2) * advantages).mean()
print("ppo_loss", round(float(ppo_loss), 4))
logp_actor = torch.tensor([[-1.0, -1.5, -0.7], [-0.8, -1.2, -1.1]])
logp_ref = torch.tensor([[-1.1, -1.3, -0.9], [-0.8, -1.0, -1.4]])
raw_rm_reward = torch.tensor([1.0, -0.2])
final_reward = raw_rm_reward - 0.02 * (logp_actor - logp_ref).sum(dim=-1)
print("final_reward", final_reward)


## Step 4: official artifact smoke check


In [ ]:
official_root = REPO / "learning" / "rlhf-classic" / "official" / "repos" / "following-instructions-human-feedback"
files = [official_root / "README.md", official_root / "model-card.md", official_root / "automatic-eval-samples" / "tldr_samples.csv"]
for file in files:
    print(file.relative_to(REPO), "exists=", file.exists())
assert all(file.exists() for file in files)


## Closed-book checks

1. List the data fields needed by SFT, reward-model training, and PPO.
2. Explain why Bradley-Terry uses only relative reward differences.
3. Explain what the frozen reference model protects against.
